# 🏪 PRAKTIKUM 15B — Simulator Kebijakan Keuntungan Toko
> **Mesin Replika: What-If Analysis dengan Regresi Linear**
---
## Alur Kerja
```
[Dataset Historis] → [EDA] → [Latih Model] → [Evaluasi] → [Ekspor JSON] → [Web Simulator]
```
## Library yang Digunakan
| Library | Fungsi |
|---|---|
| `pandas` / `numpy` | Pengolahan data skenario |
| `scikit-learn` | Melatih model regresi |
| `matplotlib` / `seaborn` | Visualisasi EDA & evaluasi |
| `json` | Ekspor model ke file .pkl (Pickle) |


## 📦 Langkah 0 — Instalasi Library

In [ ]:
# Jika Anda menjalankan ini di komputer lokal dan sudah menginstall library,
# silakan lewati atau hapus cell ini.
# !pip install numpy pandas scikit-learn matplotlib seaborn -q


## 📊 Langkah 1 — Persiapan & Pembuatan Dataset Historis

Dataset mencerminkan data penjualan toko selama beberapa periode.
Fitur kontrol (variabel independen):
- **Iklan** : Anggaran iklan dalam juta rupiah (0–50)
- **Diskon** : Besaran diskon yang diberikan dalam persen (0–50)

Target (variabel dependen):
- **Keuntungan** : Keuntungan bersih toko dalam juta rupiah

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')
# ── Gaya Visualisasi (Brutalism-inspired) ────────────────────────────────────
plt.rcParams.update({
    'font.family'       : 'monospace',
    'axes.facecolor'    : '#f5f0e8',
    'figure.facecolor'  : '#0a0a0a',
    'axes.edgecolor'    : '#0a0a0a',
    'axes.linewidth'    : 2.5,
    'axes.titlesize'    : 13,
    'axes.titleweight'  : 'bold',
    'axes.labelsize'    : 11,
    'axes.labelweight'  : 'bold',
    'axes.labelcolor'   : '#f5f0e8',
    'axes.titlecolor'   : '#ffe600',
    'xtick.color'       : '#f5f0e8',
    'ytick.color'       : '#f5f0e8',
    'grid.color'        : '#333333',
    'grid.linewidth'    : 0.8,
    'text.color'        : '#f5f0e8',
})
np.random.seed(42)
# ── Buat Dataset Historis Realistis ──────────────────────────────────────────
N = 80  # Jumlah observasi historis
iklan  = np.random.uniform(1, 50, N).round(1)   # Anggaran Iklan (Juta)
diskon = np.random.uniform(0, 50, N).round(1)   # Besaran Diskon (%)
# True relationship: Keuntungan = 20 + 3.5*iklan + 0.8*diskon + noise
# Iklan punya efek positif kuat; diskon positif tapi lebih kecil
noise        = np.random.normal(0, 8, N)
keuntungan   = 20 + 3.5 * iklan + 0.8 * diskon + noise
keuntungan   = keuntungan.round(2)
df = pd.DataFrame({
    'Iklan_Juta' : iklan,
    'Diskon_Pct' : diskon,
    'Keuntungan' : keuntungan
})
# Simpan dataset sebagai CSV
df.to_csv('dataset_toko.csv', index=False)
print(f'Dataset berhasil dibuat: {N} baris')
print(f'\nStatistik Deskriptif:')
df.describe().round(2)


In [ ]:
# ── Tampilkan 10 Baris Pertama ────────────────────────────────────────────────
print('=== 10 BARIS PERTAMA DATASET ===')
df.head(10)

## 🔍 Langkah 2 — Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('EDA — DISTRIBUSI VARIABEL', fontsize=14, fontweight='bold',
             color='#ffe600', y=1.02)

colors = ['#ffe600', '#ff2d2d', '#00d26a']
cols   = ['Iklan_Juta', 'Diskon_Pct', 'Keuntungan']
labels = ['Anggaran Iklan (Juta)', 'Besaran Diskon (%)', 'Keuntungan (Juta)']

for ax, col, label, color in zip(axes, cols, labels, colors):
    ax.hist(df[col], bins=15, color=color, edgecolor='#0a0a0a', linewidth=1.5)
    ax.set_title(label.upper(), pad=10)
    ax.set_facecolor('#1a1a1a')
    ax.grid(True, axis='y', alpha=0.4)
    mean_val = df[col].mean()
    ax.axvline(mean_val, color='white', linestyle='--', linewidth=1.8,
               label=f'Mean={mean_val:.1f}')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('eda_distribusi.png', dpi=150, bbox_inches='tight',
            facecolor='#0a0a0a')
plt.show()
print('✅ Plot EDA disimpan: eda_distribusi.png')

In [ ]:
# ── Correlation Heatmap ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0a0a0a')

# Heatmap Korelasi
corr = df.corr(numeric_only=True)
mask = np.zeros_like(corr)
mask[np.triu_indices_from(mask)] = True

sns.heatmap(
    corr, ax=axes[0], annot=True, fmt='.3f',
    cmap='YlOrRd', linewidths=2, linecolor='#0a0a0a',
    annot_kws={'size': 12, 'weight': 'bold', 'color': '#0a0a0a'},
    cbar_kws={'shrink': 0.8}
)
axes[0].set_title('KORELASI ANTAR VARIABEL', color='#ffe600', fontweight='bold')
axes[0].set_facecolor('#1a1a1a')
axes[0].tick_params(colors='#f5f0e8')

# Scatter: Iklan vs Keuntungan
axes[1].scatter(df['Iklan_Juta'], df['Keuntungan'],
                color='#ffe600', edgecolors='#0a0a0a',
                linewidths=1.2, s=60, alpha=0.85)
axes[1].set_title('IKLAN vs KEUNTUNGAN', color='#ffe600', fontweight='bold')
axes[1].set_xlabel('Anggaran Iklan (Juta)')
axes[1].set_ylabel('Keuntungan (Juta)')
axes[1].set_facecolor('#1a1a1a')
axes[1].grid(True, alpha=0.3)

# Trend line
z  = np.polyfit(df['Iklan_Juta'], df['Keuntungan'], 1)
p  = np.poly1d(z)
xs = np.linspace(df['Iklan_Juta'].min(), df['Iklan_Juta'].max(), 100)
axes[1].plot(xs, p(xs), color='#ff2d2d', linewidth=2.5, linestyle='--',
             label='Tren Linear')
axes[1].legend()

plt.tight_layout()
plt.savefig('eda_korelasi.png', dpi=150, bbox_inches='tight',
            facecolor='#0a0a0a')
plt.show()
print('✅ Plot korelasi disimpan: eda_korelasi.png')

## 🏋️ Langkah 3 — Pelatihan Model (Mesin Replika)

Model regresi linear dilatih menggunakan **scikit-learn** dengan pembagian data 80/20 (train/test).

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score
)

# ── Pisahkan Fitur & Target ───────────────────────────────────────────────────
X = df[['Iklan_Juta', 'Diskon_Pct']].values
y = df['Keuntungan'].values

# ── Split Train / Test (80 / 20) ─────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ── Latih Model ───────────────────────────────────────────────────────────────
model = LinearRegression()
model.fit(X_train, y_train)

# ── Koefisien ─────────────────────────────────────────────────────────────────
b0 = model.intercept_
b1, b2 = model.coef_

print('=' * 55)
print('  HASIL PELATIHAN MODEL — LINEAR REGRESSION')
print('=' * 55)
print(f'  Intercept (β₀)      : {b0:.4f}')
print(f'  Koef. Iklan (β₁)    : {b1:.4f}  per Juta')
print(f'  Koef. Diskon (β₂)   : {b2:.4f}  per %')
print()
print(f'  Persamaan Model:')
print(f'  Ŷ = {b0:.2f} + {b1:.2f}·Iklan + {b2:.2f}·Diskon')
print('=' * 55)

## 📈 Langkah 4 — Evaluasi Model

In [ ]:
# ── Prediksi & Metrik ─────────────────────────────────────────────────────────
y_pred_train = model.predict(X_train)
y_pred_test  = model.predict(X_test)

r2_train  = r2_score(y_train, y_pred_train)
r2_test   = r2_score(y_test,  y_pred_test)
mse_test  = mean_squared_error(y_test, y_pred_test)
rmse_test = np.sqrt(mse_test)
mae_test  = mean_absolute_error(y_test, y_pred_test)

print('=' * 55)
print('  METRIK EVALUASI MODEL')
print('=' * 55)
print(f'  R² (Train)           : {r2_train:.4f}')
print(f'  R² (Test)            : {r2_test:.4f}')
print(f'  RMSE (Test)          : {rmse_test:.4f} Juta')
print(f'  MAE  (Test)          : {mae_test:.4f} Juta')
print('=' * 55)

if r2_test >= 0.80:
    print('  ✅ Model LAYAK digunakan sebagai Mesin Replika')
elif r2_test >= 0.60:
    print('  ⚠️  Model CUKUP — pertimbangkan fitur tambahan')
else:
    print('  ❌ Model LEMAH — perlu rekayasa fitur lebih lanjut')

In [ ]:
# ── Plot Evaluasi: Actual vs Predicted & Residual ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0a0a0a')
fig.suptitle('EVALUASI MODEL', fontsize=14, fontweight='bold',
             color='#ffe600')

# Panel 1: Actual vs Predicted
axes[0].scatter(y_test, y_pred_test, color='#ffe600',
                edgecolors='#0a0a0a', linewidths=1.2, s=70, alpha=0.9,
                label='Test Data')
lims = [min(y_test.min(), y_pred_test.min()) - 5,
        max(y_test.max(), y_pred_test.max()) + 5]
axes[0].plot(lims, lims, 'w--', linewidth=1.8, label='Prediksi Sempurna')
axes[0].set_xlim(lims); axes[0].set_ylim(lims)
axes[0].set_title(f'AKTUAL vs PREDIKSI  (R²={r2_test:.3f})',
                  color='#ffe600', fontweight='bold')
axes[0].set_xlabel('Aktual (Juta)'); axes[0].set_ylabel('Prediksi (Juta)')
axes[0].set_facecolor('#1a1a1a'); axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=8)
axes[0].text(0.05, 0.92, f'RMSE={rmse_test:.2f}  MAE={mae_test:.2f}',
             transform=axes[0].transAxes, fontsize=9, color='#aaa')

# Panel 2: Residual Distribution
residuals = y_test - y_pred_test
axes[1].hist(residuals, bins=12, color='#ff2d2d',
             edgecolor='#0a0a0a', linewidth=1.5)
axes[1].axvline(0, color='#ffe600', linewidth=2, linestyle='--',
                label='Residual = 0')
axes[1].set_title('DISTRIBUSI RESIDUAL', color='#ffe600', fontweight='bold')
axes[1].set_xlabel('Residual (Aktual − Prediksi)')
axes[1].set_ylabel('Frekuensi')
axes[1].set_facecolor('#1a1a1a'); axes[1].grid(True, axis='y', alpha=0.3)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('evaluasi_model.png', dpi=150, bbox_inches='tight',
            facecolor='#0a0a0a')
plt.show()
print('✅ Plot evaluasi disimpan: evaluasi_model.png')

## ⚙️ Langkah 5 — Menetapkan Baseline & Uji Simulasi

In [ ]:
import numpy as np

# ── Baseline Condition ────────────────────────────────────────────────────────
BASELINE_IKLAN  = 10   # Juta
BASELINE_DISKON = 10   # %

baseline_input = np.array([[BASELINE_IKLAN, BASELINE_DISKON]])
baseline_pred  = model.predict(baseline_input)[0]

print(f'Kondisi Baseline  : Iklan={BASELINE_IKLAN} Jt, Diskon={BASELINE_DISKON}%')
print(f'Prediksi Baseline : Rp {baseline_pred:.2f} Juta')
print()

# ── Fungsi Simulasi What-If ───────────────────────────────────────────────────
def run_simulation(new_iklan, new_diskon):
    """Menerima input intervensi dan mengembalikan prediksi + delta."""
    intervention_input = np.array([[new_iklan, new_diskon]])
    prediction         = model.predict(intervention_input)[0]
    delta_y            = prediction - baseline_pred
    efficiency         = (delta_y / baseline_pred) * 100 if baseline_pred != 0 else 0
    return round(prediction, 2), round(delta_y, 2), round(efficiency, 2)

# ── Uji Beberapa Skenario ─────────────────────────────────────────────────────
skenario_list = [
    (BASELINE_IKLAN, BASELINE_DISKON, 'Baseline'),
    (20, 10, 'Tingkatkan Iklan'),
    (10, 30, 'Tingkatkan Diskon'),
    (30, 25, 'Agresif — Iklan + Diskon'),
    (5,  5,  'Hemat Anggaran'),
    (50, 50, 'Maksimum Semua'),
]

results = []
print(f'{"Skenario":<28} {"Iklan":>6} {"Diskon":>7} {"Prediksi":>10} {"Delta":>9} {"Efisiensi":>10}')
print('-' * 75)
for iklan, diskon, label in skenario_list:
    pred, delta, eff = run_simulation(iklan, diskon)
    results.append({'Skenario': label, 'Iklan': iklan, 'Diskon': diskon,
                    'Prediksi': pred, 'Delta': delta, 'Efisiensi%': eff})
    sign = '+' if delta >= 0 else ''
    print(f'{label:<28} {iklan:>5}Jt {diskon:>5}%  {pred:>9.2f}Jt {sign}{delta:>7.2f}Jt {sign}{eff:>8.2f}%')

df_results = pd.DataFrame(results)
print()
print('✅ Simulasi What-If selesai.')

In [ ]:
# ── Visualisasi Perbandingan Skenario ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0a0a0a')
fig.suptitle('ANALISIS SKENARIO WHAT-IF', fontsize=14,
             fontweight='bold', color='#ffe600')

labels  = [r['Skenario'] for r in results]
preds   = [r['Prediksi'] for r in results]
deltas  = [r['Delta']    for r in results]
bar_colors = ['#555555' if d == 0 else ('#00d26a' if d > 0 else '#ff2d2d')
              for d in deltas]

# Panel 1: Prediksi per skenario
bars1 = axes[0].bar(labels, preds, color='#ffe600', edgecolor='#0a0a0a',
                    linewidth=2)
axes[0].bar(labels[:1], preds[:1], color='#0a0a0a', edgecolor='#ffe600',
            linewidth=2.5, label='Baseline')
axes[0].set_title('PREDIKSI KEUNTUNGAN PER SKENARIO',
                  color='#ffe600', fontweight='bold')
axes[0].set_ylabel('Keuntungan (Juta)')
axes[0].set_facecolor('#1a1a1a')
axes[0].grid(True, axis='y', alpha=0.3)
axes[0].tick_params(axis='x', rotation=30)
for bar, val in zip(bars1, preds):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}', ha='center', va='bottom', fontsize=8,
                 color='#ffe600', fontweight='bold')

# Panel 2: Delta per skenario
bars2 = axes[1].bar(labels, deltas, color=bar_colors,
                    edgecolor='#0a0a0a', linewidth=2)
axes[1].axhline(0, color='white', linewidth=1.5, linestyle='--')
axes[1].set_title('DELTA vs BASELINE PER SKENARIO',
                  color='#ffe600', fontweight='bold')
axes[1].set_ylabel('Δ Keuntungan (Juta)')
axes[1].set_facecolor('#1a1a1a')
axes[1].grid(True, axis='y', alpha=0.3)
axes[1].tick_params(axis='x', rotation=30)
pos_patch = mpatches.Patch(color='#00d26a', label='Positif (↑)')
neg_patch = mpatches.Patch(color='#ff2d2d', label='Negatif (↓)')
axes[1].legend(handles=[pos_patch, neg_patch], fontsize=8)
for bar, val in zip(bars2, deltas):
    offset = 0.5 if val >= 0 else -2
    sign   = '+' if val >= 0 else ''
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + offset,
                 f'{sign}{val:.1f}', ha='center', va='bottom',
                 fontsize=8, color='white', fontweight='bold')

plt.tight_layout()
plt.savefig('analisis_skenario.png', dpi=150, bbox_inches='tight',
            facecolor='#0a0a0a')
plt.show()
print('✅ Plot skenario disimpan: analisis_skenario.png')

## 💾 Langkah 6 — Ekspor Model ke PKL (joblib)
File `model_bisnis.pkl` akan dimuat oleh **Web Simulator** (`index.html`).
Langkah:
1. Jalankan sel ini
2. Download file `model_bisnis.pkl`
3. Taruh di folder yang sama dengan `index.html`
4. Klik **MUAT MODEL (.pkl)** di web simulator


In [ ]:
# ── Ekspor Model dengan Joblib (Sesuai Modul Dosen) ────────
model_filename = 'model_bisnis.pkl'
joblib.dump(model, model_filename)

print('=' * 55)
print(f'  ✅ MODEL BERHASIL DISIMPAN → {model_filename}')
print('=' * 55)
print('File ini berisi keseluruhan objek LinearRegression()')
print('yang siap dipanggil di Streamlit via joblib.load()')


In [ ]:
# ── Download file (Google Colab only) ────────────────────────────────────────
try:
    from google.colab import files
    files.download('model_bisnis.pkl')
    files.download('dataset_toko.csv')
    print('⬇️  File sedang diunduh...')
except ImportError:
    print('ℹ️  Bukan Google Colab — file tersimpan di direktori kerja:')
    import os
    print(f'   {os.path.abspath(model_filename)}')


## ✅ Rangkuman Hasil Praktikum
| Komponen | Deskripsi |
|---|---|
| **Dataset** | 80 observasi historis (Iklan, Diskon → Keuntungan) |
| **Model** | Linear Regression via scikit-learn |
| **Split** | 80% Train / 20% Test |
| **Persamaan** | `Ŷ = β₀ + β₁·Iklan + β₂·Diskon` |
| **Output** | `model_bisnis.pkl` → dimuat Web Simulator |
| **Simulasi** | 6 skenario What-If diuji dan divisualisasikan |
### 🎯 Kesimpulan
> Model regresi linear berhasil dilatih sebagai **Mesin Replika** yang memprediksi keuntungan toko berdasarkan variabel kontrol (Iklan & Diskon). Dengan menggunakan `model_bisnis.pkl`, Web Simulator dapat menjalankan analisis *What-If* secara interaktif tanpa memerlukan backend Python — cukup browser saja.
---
*Praktikum 15B — Simulator Kebijakan Berbasis Model ML*
